In [ ]:
##############################################################################
# Cell 0 — Setup, load everything, robust path resolution (fixed from the start)
##############################################################################
from pathlib import Path
import json, os, sys, re, shutil
import numpy as np, torch, torch.nn as nn, pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, confusion_matrix, classification_report

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
print(f"Device: {DEVICE}")

def find_dir(*fragments):
    for frag in fragments:
        m = [p for p in Path("/kaggle/input").rglob("*") if p.is_dir() and frag in p.name.lower()]
        if m: return m[0]
    return None

KAN_QWEN = find_dir("kan_qwen", "kan-qwen")
KAN_LLAMA = find_dir("kan_llama", "llama-kan")
KNOWN_IMG_DIR = find_dir("known")
NEW_IMG_DIR = find_dir("new")

for name, p in [("KAN_QWEN", KAN_QWEN), ("KAN_LLAMA", KAN_LLAMA),
                ("KNOWN_IMG_DIR", KNOWN_IMG_DIR), ("NEW_IMG_DIR", NEW_IMG_DIR)]:
    print(f"{'✅' if p else '❌'} {name}: {p}")

# FINAL_BASE: don't trust keyword-guessing find_dir for this one — search directly
# for the one file we actually need out of it, then derive paths from there.
_human_csv_matches = list(Path("/kaggle/input").rglob("human_queries_cleaned_TRUE_FINAL.csv"))
assert _human_csv_matches, "kan_route_TRULY_FINAL_v2 zip not attached — go to Add Data and attach it."
FINAL_BASE = _human_csv_matches[0].parent.parent.parent
human_df = pd.read_csv(_human_csv_matches[0])
stale_cols = [c for c in human_df.columns if c.endswith("_x") or c.endswith("_y")]
human_df = human_df.drop(columns=stale_cols) if stale_cols else human_df
print(f"✅ FINAL_BASE resolved via: {_human_csv_matches[0]}")
print(f"✅ Human benchmark: {len(human_df)} rows")

# known/new subfolder check — only fix if they're genuinely nested; otherwise leave as-is
if KNOWN_IMG_DIR == NEW_IMG_DIR:
    subdirs = {p.name.lower(): p for p in KNOWN_IMG_DIR.iterdir() if p.is_dir()}
    if "known" in subdirs and "new" in subdirs:
        KNOWN_IMG_DIR, NEW_IMG_DIR = subdirs["known"], subdirs["new"]
        print(f"✅ Resolved nested subfolders — KNOWN_IMG_DIR: {KNOWN_IMG_DIR}")
        print(f"✅ Resolved nested subfolders — NEW_IMG_DIR:   {NEW_IMG_DIR}")
    else:
        n_jpg = len(list(KNOWN_IMG_DIR.glob("*.jpg")))
        print(f"⚠️  {n_jpg} jpgs sit flat under {KNOWN_IMG_DIR} — using it for both.")

OUT = Path("/kaggle/working/part4_final")
for sub in ["", "part4_predictions", "confusion_matrices", "tables", "figures", "checkpoints_by_seed"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

if not Path("/kaggle/working/fast-kan").exists():
    os.system("git clone https://github.com/ZiyaoLi/fast-kan.git /kaggle/working/fast-kan")
sys.path.append("/kaggle/working/fast-kan")
print("\n✅ scaffold ready")

# Load raw data
with open(KAN_QWEN / "augmented_dataset_final.json") as f: qwen_data = json.load(f)
with open(KAN_LLAMA / "raw_dataset_FINAL.json") as f: llama_data = json.load(f)

qwen_text = np.load(KAN_QWEN / "aug_text_embeddings.npy")
qc = np.load(KAN_QWEN / "clip_features.npz", allow_pickle=True)
qwen_clip = {int(k): v for k, v in qc[qc.files[0]].item().items()}
qwen_clip_vecs = np.stack([qwen_clip[int(r["img_id"])] for r in qwen_data])
X_qwen_896_orig = np.concatenate([qwen_text, qwen_clip_vecs], axis=1).astype(np.float32)

llama_text = np.load(KAN_LLAMA / "text_embeddings.npy")
lc = np.load(KAN_LLAMA / "clip_features.npz", allow_pickle=True)
llama_clip = {int(k): v for k, v in lc[lc.files[0]].item().items()}
llama_clip_vecs = np.stack([llama_clip[int(r["img_id"])] for r in llama_data])
X_llama_896 = np.concatenate([llama_text, llama_clip_vecs], axis=1).astype(np.float32)

print(f"Qwen (original, pre-balancing): {X_qwen_896_orig.shape}")
print(f"Llama: {X_llama_896.shape}")


##############################################################################
# Item 1 — Shared-28 taxonomy (matting excluded)
##############################################################################
qwen_classes_all = sorted(set(r["tool"] for r in qwen_data))
llama_classes_all = sorted(set(r["tool"] for r in llama_data))
shared_29_raw = sorted(set(qwen_classes_all) & set(llama_classes_all))
assert len(shared_29_raw) == 29, f"Expected 29, got {len(shared_29_raw)}"

shared_28 = sorted([c for c in shared_29_raw if c != "matting"])
assert len(shared_28) == 28 and "matting" not in shared_28
shared28_set = set(shared_28)

shared_28_mapping = {
    "class_to_local_id": {name: i for i, name in enumerate(shared_28)},
    "local_id_to_class": {i: name for i, name in enumerate(shared_28)},
}
local_id_to_class = shared_28_mapping["local_id_to_class"]

with open(OUT / "shared_28_class_mapping.json", "w") as f:
    json.dump(shared_28_mapping, f, indent=2)
with open(OUT / "shared28_taxonomy.txt", "w") as f:
    f.write("\n".join(f"{i}: {name}" for i, name in enumerate(shared_28)))

llama_matting_count = sum(1 for r in llama_data if r["tool"] == "matting")
print(f"✅ Shared-28 taxonomy built: {len(shared_28)} classes, matting excluded "
      f"(Llama had only {llama_matting_count} matting sample)")
print(shared_28)


##############################################################################
# Item 2 — Human Shared-28 evaluation set
##############################################################################
def filter_acceptable_labels_to_shared28(label_str, shared28_set):
    if pd.isna(label_str) or str(label_str).strip() == "":
        return ""
    labels = [x.strip() for x in str(label_str).split(",")]
    return ", ".join([l for l in labels if l in shared28_set])

human_shared28 = human_df[human_df["tool_name"].isin(shared28_set)].copy()
human_shared28["acceptable_tool_labels"] = human_shared28["acceptable_tool_labels"].apply(
    lambda x: filter_acceptable_labels_to_shared28(x, shared28_set)
)
human_shared28.to_csv(OUT / "human_shared28.csv", index=False)

has_multi = human_shared28["acceptable_tool_labels"].astype(str).str.strip() != ""
strict_true = human_shared28["strict_eval"] == True
human_strict_mask = strict_true & ~has_multi
human_ambig_mask = strict_true & has_multi

print(f"Total examples: {len(human_shared28)}")
print(f"Strict single-label: {human_strict_mask.sum()}")
print(f"Strict ambiguous (multi-label): {human_ambig_mask.sum()}")
print(f"Excluded (strict_eval=False): {(~strict_true).sum()}")
print(f"Represented classes: {human_shared28['tool_name'].nunique()} / 28")
class_counts = human_shared28["tool_name"].value_counts()
print(f"Min support/class: {class_counts.min()}  Max support/class: {class_counts.max()}")


##############################################################################
# Item 2.5 — Corrective augmentation for coverage-starved classes
# (per-image-unique phrasing, so images stay independent groups — this is the
# fixed version; if any Shared-28 class has < 100 raw samples it needs this)
##############################################################################
qwen_28_counts_raw = pd.Series([r["tool"] for r in qwen_data if r["tool"] in shared28_set]).value_counts()
starved_in_shared28 = qwen_28_counts_raw[qwen_28_counts_raw < 100].sort_values()
STARVED_FIX_CLASSES = list(starved_in_shared28.index)
print("Shared-28 classes below 100 raw samples:", STARVED_FIX_CLASSES if STARVED_FIX_CLASSES else "none")

if STARVED_FIX_CLASSES:
    TARGET_PER_CLASS = 45
    from sentence_transformers import SentenceTransformer
    minilm = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

    PERSON_PROXY_TOOLS = {"person_detection", "action_recognition", "age_gender_estimation"}
    person_img_ids = sorted(set(r["img_id"] for r in qwen_data if r["tool"] in PERSON_PROXY_TOOLS))
    print(f"Candidate person-containing images available: {len(person_img_ids)}")

    BASE_TEMPLATES = {
        "pose_estimation": [
            "Estimate the body pose of the person in this {noun}.",
            "What position is this person's body in, based on the {noun}?",
            "Map the skeletal keypoints of the person shown in the {noun}.",
            "Describe the pose this person is striking in the {noun}.",
            "Identify the joint locations and posture of the person visible in the {noun}.",
        ],
        "face_detection": [
            "Detect all faces in this {noun}.",
            "Locate every face visible in this {noun}.",
            "Find and mark the faces present in the {noun}.",
            "How many faces can you identify in this {noun}?",
            "Draw bounding boxes around each face in the {noun}.",
        ],
        "facial_expression_recognition": [
            "What facial expression is this person showing in the {noun}?",
            "Identify the emotion on this person's face in the {noun}.",
            "Is the person smiling, frowning, or neutral in this {noun}?",
            "Classify the facial expression visible in the {noun}.",
            "What mood does this person's face convey in the {noun}?",
        ],
    }
    OPENERS = ["", "Please ", "Could you ", "I'd like you to ", "Go ahead and ",
               "Take a look and ", "Kindly ", "Now, ", "Next, ", "Also, ", "For this one, ", "Here, "]
    NOUNS = ["image", "photo", "picture", "photograph", "snapshot", "frame",
             "shot", "still", "scene", "visual", "capture", "graphic"]

    def normalize_q(q): return re.sub(r"[^\w\s]", "", str(q).lower()).strip()

    def make_unique_query(base_template, img_rank, class_rank):
        opener = OPENERS[img_rank % len(OPENERS)]
        noun = NOUNS[(img_rank + class_rank) % len(NOUNS)]
        text = base_template.format(noun=noun)
        return (opener + text[0].lower() + text[1:]) if opener else text

    rng = np.random.RandomState(SEED)
    new_rows, new_texts, provenance_log = [], [], []

    for cls_rank, cls in enumerate(STARVED_FIX_CLASSES):
        if cls not in BASE_TEMPLATES:
            print(f"⚠️  No templates defined for '{cls}' — skipping, add templates manually if needed")
            continue
        existing_imgs_for_cls = set(r["img_id"] for r in qwen_data if r["tool"] == cls)
        pool = [i for i in person_img_ids if i in qwen_clip]
        rng.shuffle(pool)
        n_templates = len(BASE_TEMPLATES[cls])
        n_images_needed = -(-TARGET_PER_CLASS // n_templates)
        chosen_imgs = pool[:n_images_needed]
        added, seen_texts = 0, set()
        for img_rank, img_id in enumerate(chosen_imgs):
            for tmpl in BASE_TEMPLATES[cls]:
                if added >= TARGET_PER_CLASS:
                    break
                q = make_unique_query(tmpl, img_rank, cls_rank)
                assert normalize_q(q) not in seen_texts, f"Collision for {cls}"
                seen_texts.add(normalize_q(q))
                new_rows.append({"img_id": img_id, "tool": cls, "query": q,
                                  "objects": None, "source": "manual_corrective_augmentation"})
                new_texts.append(q)
                added += 1
            if added >= TARGET_PER_CLASS:
                break
        provenance_log.append(dict(tool=cls, images_used=len(chosen_imgs), rows_added=added,
                                    previously_existing=len(existing_imgs_for_cls)))
        print(f"  {cls}: +{added} rows across {len(chosen_imgs)} images, "
              f"{len(seen_texts)} distinct strings (had {len(existing_imgs_for_cls)} images before)")

    new_text_embs = minilm.encode(new_texts, batch_size=64, convert_to_numpy=True,
                                   normalize_embeddings=False).astype(np.float32)
    new_clip_vecs = np.stack([qwen_clip[r["img_id"]] for r in new_rows]).astype(np.float32)
    X_new = np.concatenate([new_text_embs, new_clip_vecs], axis=1).astype(np.float32)

    qwen_data = qwen_data + new_rows
    X_qwen_896_orig = np.concatenate([X_qwen_896_orig, X_new], axis=0)
    assert len(qwen_data) == X_qwen_896_orig.shape[0]
    print(f"✅ qwen_data patched: {len(qwen_data)} rows total (+{len(new_rows)} corrective)")

    with open(OUT / "qwen_corrective_augmentation_provenance.md", "w") as f:
        f.write("# Qwen Corrective Augmentation — Provenance\n\n")
        f.write("**Manually template-authored augmentation, NOT teacher-model-generated.** "
                f"Fixes coverage-starved Shared-28 classes: {STARVED_FIX_CLASSES}. "
                "Per-image phrasing is varied (opener + noun choice) so each image's rows form "
                "an independent group under leakage-safe grouping, rather than merging into one "
                "atomic block that a single split partition would consume entirely.\n\n")
        f.write("Images reused from existing corpus images already labeled with a "
                "person-indicating tool, as a proxy for 'contains a person'. CLIP image "
                "embeddings pulled from the existing cache (no new encode_image() calls); "
                "only MiniLM text embeddings are freshly computed.\n\n")
        f.write(pd.DataFrame(provenance_log).to_markdown(index=False) + "\n")
        f.write("\n**Disclosure requirement:** any Part 4 result touching these classes must "
                "cite this file in Limitations.\n")
    print("✅ qwen_corrective_augmentation_provenance.md written")
else:
    print("✅ No Shared-28 classes under-supported — proceeding with X_qwen_896_orig unmodified")


##############################################################################
# Items 3-4 — Leakage-safe Shared-28 splits + audit
##############################################################################
def normalize_q(q): return re.sub(r"[^\w\s]", "", str(q).lower()).strip()

class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x: self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry: self.parent[rx] = ry

def build_groups(records):
    n = len(records); uf = UnionFind(n)
    norm_to_idx, imgtool_to_idx = {}, {}
    for i, r in enumerate(records):
        norm = normalize_q(r["query"])
        if norm in norm_to_idx: uf.union(i, norm_to_idx[norm])
        else: norm_to_idx[norm] = i
    for i, r in enumerate(records):
        key = (r["img_id"], r["tool"])
        if key in imgtool_to_idx: uf.union(i, imgtool_to_idx[key])
        else: imgtool_to_idx[key] = i
    groups = {}
    for i in range(n): groups.setdefault(uf.find(i), []).append(i)
    return list(groups.values())

def group_split(groups, y_local, seed=SEED, train_frac=0.80, val_frac=0.10):
    rng = np.random.RandomState(seed)
    group_order = sorted(range(len(groups)), key=lambda gi: len(groups[gi]))
    used = set(); val_idx, test_idx = [], []
    all_classes = set(y_local.tolist())
    def smallest_avail(cls, excl):
        c = [(gi, len(groups[gi])) for gi in group_order
             if not any(i in excl for i in groups[gi]) and any(y_local[i] == cls for i in groups[gi])]
        return min(c, key=lambda x: x[1])[0] if c else None
    for cls in all_classes:
        if any(y_local[i] == cls for i in val_idx): continue
        gi = smallest_avail(cls, used)
        if gi is not None: val_idx.extend(groups[gi]); used.update(groups[gi])
    for cls in all_classes:
        if any(y_local[i] == cls for i in test_idx): continue
        gi = smallest_avail(cls, used)
        if gi is not None: test_idx.extend(groups[gi]); used.update(groups[gi])
    remaining = [gi for gi in group_order if not any(i in used for i in groups[gi])]
    rng.shuffle(remaining)
    n_tot = len(y_local); n_tr_t = int(train_frac * n_tot); n_va_t = int(val_frac * n_tot)
    train_idx = []
    for gi in remaining:
        idxs = groups[gi]
        if len(train_idx) < n_tr_t: train_idx.extend(idxs)
        elif len(val_idx) < n_va_t: val_idx.extend(idxs)
        else: test_idx.extend(idxs)
    return train_idx, val_idx, test_idx

def verify_leakage(idx_tr, idx_va, idx_te, records, name):
    def norms(idxs): return set(normalize_q(records[i]["query"]) for i in idxs)
    def imgtool(idxs): return set((records[i]["img_id"], records[i]["tool"]) for i in idxs)
    ntr, nva, nte = norms(idx_tr), norms(idx_va), norms(idx_te)
    itr, iva, ite = imgtool(idx_tr), imgtool(idx_va), imgtool(idx_te)
    r = {"norm_tr_va": len(ntr & nva), "norm_tr_te": len(ntr & nte), "norm_va_te": len(nva & nte),
         "imgtool_tr_va": len(itr & iva), "imgtool_tr_te": len(itr & ite), "imgtool_va_te": len(iva & ite)}
    print(f"\n=== {name} leakage ===")
    for k, v in r.items(): print(f"  {'✅' if v == 0 else '❌'} {k}: {v}")
    return r, all(v == 0 for v in r.values())

qwen_28_idx = [i for i, r in enumerate(qwen_data) if r["tool"] in shared28_set]
llama_28_idx = [i for i, r in enumerate(llama_data) if r["tool"] in shared28_set]
qwen_28_records = [qwen_data[i] for i in qwen_28_idx]
llama_28_records = [llama_data[i] for i in llama_28_idx]
X_qwen_28 = X_qwen_896_orig[qwen_28_idx]
X_llama_28 = X_llama_896[llama_28_idx]
y_qwen_28 = np.array([shared_28_mapping["class_to_local_id"][r["tool"]] for r in qwen_28_records], dtype=np.int64)
y_llama_28 = np.array([shared_28_mapping["class_to_local_id"][r["tool"]] for r in llama_28_records], dtype=np.int64)

print(f"Qwen Shared-28 subset: {len(qwen_28_records)} rows")
print(f"Llama Shared-28 subset: {len(llama_28_records)} rows")

qwen_groups_28 = build_groups(qwen_28_records)
llama_groups_28 = build_groups(llama_28_records)

idx_tr_Q28, idx_va_Q28, idx_te_Q28 = group_split(qwen_groups_28, y_qwen_28)
idx_tr_L28, idx_va_L28, idx_te_L28 = group_split(llama_groups_28, y_llama_28)

print(f"\nQwen-28 split: train={len(idx_tr_Q28)} val={len(idx_va_Q28)} test={len(idx_te_Q28)}")
print(f"Llama-28 split: train={len(idx_tr_L28)} val={len(idx_va_L28)} test={len(idx_te_L28)}")

q_leak, q_zero = verify_leakage(idx_tr_Q28, idx_va_Q28, idx_te_Q28, qwen_28_records, "Qwen-28")
l_leak, l_zero = verify_leakage(idx_tr_L28, idx_va_L28, idx_te_L28, llama_28_records, "Llama-28")

# Class coverage check — the thing that broke twice before, so check explicitly every time
all28 = set(range(28))
train_classes_Q, val_classes_Q, test_classes_Q = (set(y_qwen_28[i] for i in idx) for idx in (idx_tr_Q28, idx_va_Q28, idx_te_Q28))
train_classes_L, val_classes_L, test_classes_L = (set(y_qwen_28[i] for i in idx) if False else set(y_llama_28[i] for i in idx) for idx in (idx_tr_L28, idx_va_L28, idx_te_L28))

print(f"\nQwen  — missing from train: {[local_id_to_class[c] for c in sorted(all28 - train_classes_Q)]}")
print(f"Qwen  — missing from val:   {[local_id_to_class[c] for c in sorted(all28 - val_classes_Q)]}")
print(f"Qwen  — missing from test:  {[local_id_to_class[c] for c in sorted(all28 - test_classes_Q)]}")
print(f"Llama — missing from train: {[local_id_to_class[c] for c in sorted(all28 - train_classes_L)]}")
print(f"Llama — missing from val:   {[local_id_to_class[c] for c in sorted(all28 - val_classes_L)]}")
print(f"Llama — missing from test:  {[local_id_to_class[c] for c in sorted(all28 - test_classes_L)]}")

with open(OUT / "shared28_split_leakage_audit.md", "w") as f:
    f.write("# Shared-28 Split Leakage + Coverage Audit\n\n## Qwen-28\n\n")
    for k, v in q_leak.items(): f.write(f"- {k}: {v}\n")
    f.write(f"\nLEAKAGE FREE: {q_zero}\n")
    f.write(f"Missing from train: {[local_id_to_class[c] for c in sorted(all28 - train_classes_Q)]}\n")
    f.write(f"Missing from val:   {[local_id_to_class[c] for c in sorted(all28 - val_classes_Q)]}\n")
    f.write(f"Missing from test:  {[local_id_to_class[c] for c in sorted(all28 - test_classes_Q)]}\n\n")
    f.write("## Llama-28\n\n")
    for k, v in l_leak.items(): f.write(f"- {k}: {v}\n")
    f.write(f"\nLEAKAGE FREE: {l_zero}\n")
    f.write(f"Missing from train: {[local_id_to_class[c] for c in sorted(all28 - train_classes_L)]}\n")
    f.write(f"Missing from val:   {[local_id_to_class[c] for c in sorted(all28 - val_classes_L)]}\n")
    f.write(f"Missing from test:  {[local_id_to_class[c] for c in sorted(all28 - test_classes_L)]}\n")

assert q_zero and l_zero, "STOP: leakage detected — do not continue (Item 4)"
if train_classes_Q != all28 or train_classes_L != all28:
    print("\n⚠️  STOP before training: at least one class is missing from a TRAIN split. "
          "A model cannot learn a class it never sees. Check the printout above and fix "
          "before proceeding to Items 5+.")
else:
    print("\n✅ Leakage-free AND full 28/28 class coverage in train/val/test for both teachers "
          "— safe to proceed to Item 5")

In [ ]:
##############################################################################
# Diagnostic — why is 3d_reconstruction missing from Llama's train split?
##############################################################################
llama_3dr_records = [r for r in llama_data if r["tool"] == "3d_reconstruction"]
llama_3dr_imgs = set(r["img_id"] for r in llama_3dr_records)
print(f"Llama '3d_reconstruction' raw samples: {len(llama_3dr_records)}")
print(f"Distinct images: {len(llama_3dr_imgs)}")

llama_3dr_groups = build_groups(llama_3dr_records)
print(f"Connected components (groups) among these samples: {len(llama_3dr_groups)}")
print(f"Group sizes: {sorted(len(g) for g in llama_3dr_groups)}")

# Which split did each group actually land in?
cls_id_3dr = shared_28_mapping["class_to_local_id"]["3d_reconstruction"]
llama_28_records_idx_by_orig = {orig_i: local_i for local_i, orig_i in enumerate(llama_28_idx)}
in_train = sum(1 for i in idx_tr_L28 if y_llama_28[i] == cls_id_3dr)
in_val   = sum(1 for i in idx_va_L28 if y_llama_28[i] == cls_id_3dr)
in_test  = sum(1 for i in idx_te_L28 if y_llama_28[i] == cls_id_3dr)
print(f"\nRows landing in train/val/test: {in_train} / {in_val} / {in_test}")

In [ ]:
##############################################################################
# Item 2.5b — Llama corrective augmentation for coverage-starved classes
# Run ONLY if the diagnostic confirms 3d_reconstruction has too few groups.
# Same fix pattern as Qwen's Item 2.5: per-image-unique phrasing so images
# stay independent connected components instead of merging into one block.
##############################################################################
LLAMA_STARVED_FIX_CLASSES = ["3d_reconstruction"]  # extend if diagnostic shows others
TARGET_PER_CLASS_LLAMA = 45

if "minilm" not in dir():
    from sentence_transformers import SentenceTransformer
    minilm = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

# Any image already in the Llama corpus with a cached CLIP vector is a valid
# candidate — 3d_reconstruction isn't person-specific like the Qwen classes were.
llama_candidate_imgs = sorted(set(r["img_id"] for r in llama_data if r["img_id"] in llama_clip))
print(f"Llama candidate images with cached CLIP vectors: {len(llama_candidate_imgs)}")

BASE_TEMPLATES_LLAMA = {
    "3d_reconstruction": [
        "Reconstruct the 3D structure of the scene in this {noun}.",
        "Estimate the 3D geometry of the objects shown in this {noun}.",
        "Build a 3D model from this {noun}.",
        "What is the spatial 3D layout of this {noun}?",
        "Generate a 3D representation based on this {noun}.",
    ],
}
OPENERS = ["", "Please ", "Could you ", "I'd like you to ", "Go ahead and ",
           "Take a look and ", "Kindly ", "Now, ", "Next, ", "Also, ", "For this one, ", "Here, "]
NOUNS = ["image", "photo", "picture", "photograph", "snapshot", "frame",
         "shot", "still", "scene", "visual", "capture", "graphic"]

def make_unique_query(base_template, img_rank, class_rank):
    opener = OPENERS[img_rank % len(OPENERS)]
    noun = NOUNS[(img_rank + class_rank) % len(NOUNS)]
    text = base_template.format(noun=noun)
    return (opener + text[0].lower() + text[1:]) if opener else text

rng = np.random.RandomState(SEED)
new_rows_llama, new_texts_llama, provenance_log_llama = [], [], []

for cls_rank, cls in enumerate(LLAMA_STARVED_FIX_CLASSES):
    existing_imgs_for_cls = set(r["img_id"] for r in llama_data if r["tool"] == cls)
    pool = [i for i in llama_candidate_imgs if i not in existing_imgs_for_cls]
    rng.shuffle(pool)
    n_templates = len(BASE_TEMPLATES_LLAMA[cls])
    n_images_needed = -(-TARGET_PER_CLASS_LLAMA // n_templates)
    chosen_imgs = pool[:n_images_needed]
    added, seen_texts = 0, set()
    for img_rank, img_id in enumerate(chosen_imgs):
        for tmpl in BASE_TEMPLATES_LLAMA[cls]:
            if added >= TARGET_PER_CLASS_LLAMA:
                break
            q = make_unique_query(tmpl, img_rank, cls_rank)
            assert normalize_q(q) not in seen_texts, f"Collision for {cls}"
            seen_texts.add(normalize_q(q))
            new_rows_llama.append({"img_id": img_id, "tool": cls, "query": q,
                                    "objects": None, "source": "manual_corrective_augmentation"})
            new_texts_llama.append(q)
            added += 1
        if added >= TARGET_PER_CLASS_LLAMA:
            break
    provenance_log_llama.append(dict(tool=cls, images_used=len(chosen_imgs), rows_added=added,
                                      previously_existing=len(existing_imgs_for_cls)))
    print(f"  {cls}: +{added} rows across {len(chosen_imgs)} images, "
          f"{len(seen_texts)} distinct strings (had {len(existing_imgs_for_cls)} images before)")

new_text_embs_llama = minilm.encode(new_texts_llama, batch_size=64, convert_to_numpy=True,
                                     normalize_embeddings=False).astype(np.float32)
new_clip_vecs_llama = np.stack([llama_clip[r["img_id"]] for r in new_rows_llama]).astype(np.float32)
X_new_llama = np.concatenate([new_text_embs_llama, new_clip_vecs_llama], axis=1).astype(np.float32)

llama_data = llama_data + new_rows_llama
X_llama_896 = np.concatenate([X_llama_896, X_new_llama], axis=0)
assert len(llama_data) == X_llama_896.shape[0]
print(f"✅ llama_data patched: {len(llama_data)} rows total (+{len(new_rows_llama)} corrective)")

# Sanity check: confirm images stay independent groups, no merged block
_check_records = [r for r in llama_data if r["tool"] in LLAMA_STARVED_FIX_CLASSES]
_check_groups = build_groups(_check_records)
_sizes = sorted(len(g) for g in _check_groups)
print(f"Group-size check: {len(_check_groups)} groups, sizes={_sizes}")
assert max(_sizes) <= 6, "STOP: merged block detected — phrasing not unique enough"
print("✅ No merged blocks — each image forms its own independent group")

with open(OUT / "llama_corrective_augmentation_provenance.md", "w") as f:
    f.write("# Llama Corrective Augmentation — Provenance\n\n")
    f.write("**Manually template-authored augmentation, NOT teacher-model-generated.** "
            f"Fixes coverage-starved Shared-28 classes: {LLAMA_STARVED_FIX_CLASSES}. "
            "Same per-image-unique-phrasing approach used for the analogous Qwen fix — "
            "see qwen_corrective_augmentation_provenance.md for the full rationale.\n\n")
    f.write("Images reused from existing Llama corpus images (any tool) with a cached CLIP "
            "vector already available; no new encode_image() calls were made. Only MiniLM "
            "text embeddings are freshly computed for the new query text.\n\n")
    f.write(pd.DataFrame(provenance_log_llama).to_markdown(index=False) + "\n")
    f.write("\n**Disclosure requirement:** any Part 4 result touching these classes must cite "
            "this file in Limitations.\n")
print("✅ llama_corrective_augmentation_provenance.md written")

print("\n➡️  Now re-run Items 3-4 (the split + leakage + coverage-check cell) to rebuild "
      "llama_28_idx / X_llama_28 / y_llama_28 / splits from the patched llama_data.")

In [ ]:

 
##############################################################################
# Items 3-4 — Leakage-safe Shared-28 splits + audit
##############################################################################
def normalize_q(q): return re.sub(r"[^\w\s]", "", str(q).lower()).strip()
 
class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x: self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry: self.parent[rx] = ry
 
def build_groups(records):
    n = len(records); uf = UnionFind(n)
    norm_to_idx, imgtool_to_idx = {}, {}
    for i, r in enumerate(records):
        norm = normalize_q(r["query"])
        if norm in norm_to_idx: uf.union(i, norm_to_idx[norm])
        else: norm_to_idx[norm] = i
    for i, r in enumerate(records):
        key = (r["img_id"], r["tool"])
        if key in imgtool_to_idx: uf.union(i, imgtool_to_idx[key])
        else: imgtool_to_idx[key] = i
    groups = {}
    for i in range(n): groups.setdefault(uf.find(i), []).append(i)
    return list(groups.values())
 
def group_split(groups, y_local, seed=SEED, train_frac=0.80, val_frac=0.10):
    rng = np.random.RandomState(seed)
    group_order = sorted(range(len(groups)), key=lambda gi: len(groups[gi]))
    used = set(); val_idx, test_idx = [], []
    all_classes = set(y_local.tolist())
    def smallest_avail(cls, excl):
        c = [(gi, len(groups[gi])) for gi in group_order
             if not any(i in excl for i in groups[gi]) and any(y_local[i] == cls for i in groups[gi])]
        return min(c, key=lambda x: x[1])[0] if c else None
    for cls in all_classes:
        if any(y_local[i] == cls for i in val_idx): continue
        gi = smallest_avail(cls, used)
        if gi is not None: val_idx.extend(groups[gi]); used.update(groups[gi])
    for cls in all_classes:
        if any(y_local[i] == cls for i in test_idx): continue
        gi = smallest_avail(cls, used)
        if gi is not None: test_idx.extend(groups[gi]); used.update(groups[gi])
    remaining = [gi for gi in group_order if not any(i in used for i in groups[gi])]
    rng.shuffle(remaining)
    n_tot = len(y_local); n_tr_t = int(train_frac * n_tot); n_va_t = int(val_frac * n_tot)
    train_idx = []
    for gi in remaining:
        idxs = groups[gi]
        if len(train_idx) < n_tr_t: train_idx.extend(idxs)
        elif len(val_idx) < n_va_t: val_idx.extend(idxs)
        else: test_idx.extend(idxs)
    return train_idx, val_idx, test_idx
 
def verify_leakage(idx_tr, idx_va, idx_te, records, name):
    def norms(idxs): return set(normalize_q(records[i]["query"]) for i in idxs)
    def imgtool(idxs): return set((records[i]["img_id"], records[i]["tool"]) for i in idxs)
    ntr, nva, nte = norms(idx_tr), norms(idx_va), norms(idx_te)
    itr, iva, ite = imgtool(idx_tr), imgtool(idx_va), imgtool(idx_te)
    r = {"norm_tr_va": len(ntr & nva), "norm_tr_te": len(ntr & nte), "norm_va_te": len(nva & nte),
         "imgtool_tr_va": len(itr & iva), "imgtool_tr_te": len(itr & ite), "imgtool_va_te": len(iva & ite)}
    print(f"\n=== {name} leakage ===")
    for k, v in r.items(): print(f"  {'✅' if v == 0 else '❌'} {k}: {v}")
    return r, all(v == 0 for v in r.values())
 
qwen_28_idx = [i for i, r in enumerate(qwen_data) if r["tool"] in shared28_set]
llama_28_idx = [i for i, r in enumerate(llama_data) if r["tool"] in shared28_set]
qwen_28_records = [qwen_data[i] for i in qwen_28_idx]
llama_28_records = [llama_data[i] for i in llama_28_idx]
X_qwen_28 = X_qwen_896_orig[qwen_28_idx]
X_llama_28 = X_llama_896[llama_28_idx]
y_qwen_28 = np.array([shared_28_mapping["class_to_local_id"][r["tool"]] for r in qwen_28_records], dtype=np.int64)
y_llama_28 = np.array([shared_28_mapping["class_to_local_id"][r["tool"]] for r in llama_28_records], dtype=np.int64)
 
print(f"Qwen Shared-28 subset: {len(qwen_28_records)} rows")
print(f"Llama Shared-28 subset: {len(llama_28_records)} rows")
 
qwen_groups_28 = build_groups(qwen_28_records)
llama_groups_28 = build_groups(llama_28_records)
 
idx_tr_Q28, idx_va_Q28, idx_te_Q28 = group_split(qwen_groups_28, y_qwen_28)
idx_tr_L28, idx_va_L28, idx_te_L28 = group_split(llama_groups_28, y_llama_28)
 
print(f"\nQwen-28 split: train={len(idx_tr_Q28)} val={len(idx_va_Q28)} test={len(idx_te_Q28)}")
print(f"Llama-28 split: train={len(idx_tr_L28)} val={len(idx_va_L28)} test={len(idx_te_L28)}")
 
q_leak, q_zero = verify_leakage(idx_tr_Q28, idx_va_Q28, idx_te_Q28, qwen_28_records, "Qwen-28")
l_leak, l_zero = verify_leakage(idx_tr_L28, idx_va_L28, idx_te_L28, llama_28_records, "Llama-28")
 
# Class coverage check — the thing that broke twice before, so check explicitly every time
all28 = set(range(28))
train_classes_Q, val_classes_Q, test_classes_Q = (set(y_qwen_28[i] for i in idx) for idx in (idx_tr_Q28, idx_va_Q28, idx_te_Q28))
train_classes_L, val_classes_L, test_classes_L = (set(y_qwen_28[i] for i in idx) if False else set(y_llama_28[i] for i in idx) for idx in (idx_tr_L28, idx_va_L28, idx_te_L28))
 
print(f"\nQwen  — missing from train: {[local_id_to_class[c] for c in sorted(all28 - train_classes_Q)]}")
print(f"Qwen  — missing from val:   {[local_id_to_class[c] for c in sorted(all28 - val_classes_Q)]}")
print(f"Qwen  — missing from test:  {[local_id_to_class[c] for c in sorted(all28 - test_classes_Q)]}")
print(f"Llama — missing from train: {[local_id_to_class[c] for c in sorted(all28 - train_classes_L)]}")
print(f"Llama — missing from val:   {[local_id_to_class[c] for c in sorted(all28 - val_classes_L)]}")
print(f"Llama — missing from test:  {[local_id_to_class[c] for c in sorted(all28 - test_classes_L)]}")
 
with open(OUT / "shared28_split_leakage_audit.md", "w") as f:
    f.write("# Shared-28 Split Leakage + Coverage Audit\n\n## Qwen-28\n\n")
    for k, v in q_leak.items(): f.write(f"- {k}: {v}\n")
    f.write(f"\nLEAKAGE FREE: {q_zero}\n")
    f.write(f"Missing from train: {[local_id_to_class[c] for c in sorted(all28 - train_classes_Q)]}\n")
    f.write(f"Missing from val:   {[local_id_to_class[c] for c in sorted(all28 - val_classes_Q)]}\n")
    f.write(f"Missing from test:  {[local_id_to_class[c] for c in sorted(all28 - test_classes_Q)]}\n\n")
    f.write("## Llama-28\n\n")
    for k, v in l_leak.items(): f.write(f"- {k}: {v}\n")
    f.write(f"\nLEAKAGE FREE: {l_zero}\n")
    f.write(f"Missing from train: {[local_id_to_class[c] for c in sorted(all28 - train_classes_L)]}\n")
    f.write(f"Missing from val:   {[local_id_to_class[c] for c in sorted(all28 - val_classes_L)]}\n")
    f.write(f"Missing from test:  {[local_id_to_class[c] for c in sorted(all28 - test_classes_L)]}\n")
 
assert q_zero and l_zero, "STOP: leakage detected — do not continue (Item 4)"
if train_classes_Q != all28 or train_classes_L != all28:
    print("\n⚠️  STOP before training: at least one class is missing from a TRAIN split. "
          "A model cannot learn a class it never sees. Check the printout above and fix "
          "before proceeding to Items 5+.")
else:
    print("\n✅ Leakage-free AND full 28/28 class coverage in train/val/test for both teachers "
          "— safe to proceed to Item 5")
 

In [ ]:
!pip install -q open_clip_torch

In [ ]:
##############################################################################
# Item 5 — Feature verification: confirm 896-d = MiniLM(384) + real Pixel-CLIP(512)
##############################################################################
assert X_qwen_28.shape[1] == 896 and X_llama_28.shape[1] == 896, "Feature dim must be 896"
assert qwen_text.shape[1] == 384 and llama_text.shape[1] == 384, "MiniLM dim must be 384"
assert qwen_clip_vecs.shape[1] == 512 and llama_clip_vecs.shape[1] == 512, "CLIP dim must be 512"
print(f"✅ Dimension check: Qwen 384+512={X_qwen_896_orig.shape[1]}, "
      f"Llama 384+512={X_llama_896.shape[1]}")

# Re-verify cached CLIP vectors are real pixel embeddings by recomputing encode_image()
# fresh on a sample and comparing to the cache.
import open_clip
from PIL import Image

def find_coco_image(img_id, image_dirs):
    for d in image_dirs:
        p = Path(d) / f"{img_id}.jpg"
        if p.exists():
            return p
    return None

image_dirs = [KNOWN_IMG_DIR, NEW_IMG_DIR]
caches = {**qwen_clip, **llama_clip}
candidates = [i for i in caches.keys() if find_coco_image(i, image_dirs) is not None][:15]
print(f"Provenance sample: {len(candidates)} images locatable on disk out of {len(caches)} cached IDs")
assert len(candidates) >= 5, "STOP: too few images locatable — check KNOWN_IMG_DIR/NEW_IMG_DIR paths"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai", force_quick_gelu=True
)
clip_model = clip_model.to(DEVICE).eval()

diffs = []
for img_id in candidates:
    img_path = find_coco_image(img_id, image_dirs)
    image = Image.open(img_path).convert("RGB")
    pixel_values = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        fresh = clip_model.encode_image(pixel_values)
        fresh = (fresh / fresh.norm(dim=-1, keepdim=True)).cpu().numpy()[0]
    diff = float(np.abs(fresh - caches[img_id]).max())
    diffs.append(diff)

TOLERANCE = 1e-4
provenance_result = {
    "images_tested": len(candidates),
    "images_matched": len(diffs),
    "maximum_absolute_difference": max(diffs) if diffs else None,
    "PROVENANCE_VERIFIED": bool(diffs) and len(diffs) == len(candidates) and max(diffs) < TOLERANCE,
}
print(json.dumps(provenance_result, indent=2))
assert provenance_result["PROVENANCE_VERIFIED"], "STOP: cached CLIP vectors don't match fresh encode_image() output"
print("✅ Pixel-CLIP provenance verified — cached features are real image embeddings")

with open(OUT / "pixel_clip_provenance_verification.json", "w") as f:
    json.dump(provenance_result, f, indent=2)


##############################################################################
# Item 5b — Build human Shared-28 features (MiniLM query + real Pixel-CLIP image)
##############################################################################
if "minilm" not in dir():
    from sentence_transformers import SentenceTransformer
    minilm = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

human_text_28 = minilm.encode(
    human_shared28["query"].tolist(), batch_size=256, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=False
).astype(np.float32)
assert human_text_28.shape[1] == 384

human_missing_images, human_clip_vecs = [], []
for iid in human_shared28["image_id"]:
    p = find_coco_image(iid, image_dirs)
    if p is None:
        human_missing_images.append(iid)
        human_clip_vecs.append(np.zeros(512, dtype=np.float32))
        continue
    image = Image.open(p).convert("RGB")
    pixel_values = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        f = clip_model.encode_image(pixel_values)
        f = (f / f.norm(dim=-1, keepdim=True)).cpu().numpy()[0]
    human_clip_vecs.append(f)
human_clip_vecs = np.stack(human_clip_vecs).astype(np.float32)

if human_missing_images:
    print(f"⚠️  {len(human_missing_images)} human-benchmark images not found on disk: "
          f"{human_missing_images[:20]}{'...' if len(human_missing_images) > 20 else ''}")
    with open(OUT / "human_missing_images.json", "w") as f:
        json.dump([int(x) for x in human_missing_images], f, indent=2)
    keep_mask = ~human_shared28["image_id"].isin(human_missing_images)
    human_shared28 = human_shared28[keep_mask].reset_index(drop=True)
    human_text_28 = human_text_28[keep_mask.values]
    human_clip_vecs = human_clip_vecs[keep_mask.values]
    # keep the strict/ambiguous masks in sync with the filtered rows
    has_multi = human_shared28["acceptable_tool_labels"].astype(str).str.strip() != ""
    strict_true = human_shared28["strict_eval"] == True
    human_strict_mask = strict_true & ~has_multi
    human_ambig_mask = strict_true & has_multi
    print(f"   Filtered to {len(human_shared28)} rows with resolvable images (reproducible subset logged).")
else:
    print("✅ All human-benchmark images resolved on disk — no filtering needed")

X_human_28 = np.concatenate([human_text_28, human_clip_vecs], axis=1).astype(np.float32)
y_human_28_canonical = human_shared28["tool_name"].map(shared_28_mapping["class_to_local_id"]).values.astype(np.int64)
print(f"✅ Human Shared-28 features: {X_human_28.shape}")
print(f"   Strict single-label: {human_strict_mask.sum()}, ambiguous: {human_ambig_mask.sum()}")

In [ ]:
##############################################################################
# Cell 1 (re-included, idempotent) — Architecture + helpers
##############################################################################
from fastkan import FastKAN

class KANRoute(nn.Module):
    def __init__(self, input_dim=896, proj_dim=48, hidden_dims=[48], num_tools=28, num_grids=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim)
        self.norm = nn.LayerNorm(proj_dim)
        self.kan = FastKAN(layers_hidden=[proj_dim] + hidden_dims + [num_tools], num_grids=num_grids)
    def forward(self, x): return self.kan(self.norm(self.proj(x.float())))
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class MLPBaseline(nn.Module):
    def __init__(self, input_dim=896, proj_dim=96, hidden_dims=[64, 32], num_tools=28, dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, proj_dim), nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(dropout)]
        prev = proj_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_tools))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x.float())
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class ToolRoutingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def evaluate_full_probs(model, loader):
    model.eval(); preds, labels, all_probs = [], [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            logits = model(X_b)
            probs = torch.softmax(logits, dim=-1)
            preds.extend(logits.argmax(-1).cpu().numpy())
            labels.extend(y_b.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(preds), np.array(labels), np.array(all_probs)

def train_model(model, tr, va, name, epochs=60, lr=1e-3, wd=1e-4, is_kan=False, seed=42, save_dir=None):
    torch.manual_seed(seed)
    if is_kan: lr, epochs = 5e-4, 80
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, steps_per_epoch=len(tr), epochs=epochs, pct_start=0.1)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    best, path = 0.0, (save_dir or OUT) / f"{name}_best.pt"
    for ep in range(1, epochs + 1):
        model.train()
        for X_b, y_b in tr:
            X_b, y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            opt.zero_grad(); loss = crit(model(X_b), y_b); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step()
        model.eval(); c, t = 0, 0
        with torch.no_grad():
            for X_b, y_b in va:
                X_b, y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
                c += (model(X_b).argmax(-1) == y_b).sum().item(); t += len(y_b)
        acc = c / t
        if acc > best: best = acc; torch.save(model.state_dict(), path)
        if ep % 20 == 0 or ep == 1: print(f"  [{name}] ep{ep}/{epochs} val={acc:.4f}")
    model.load_state_dict(torch.load(path))
    return best

TRAINING_CONFIG = dict(optimizer="AdamW", lr_mlp=1e-3, lr_kan=5e-4, weight_decay=1e-4,
                        batch_size=256, epochs_mlp=60, epochs_kan=80,
                        scheduler="OneCycleLR", grad_clip=1.0, checkpoint_criterion="best_val_accuracy")
print("✅ Cell 1 ready. Training config:", TRAINING_CONFIG)


##############################################################################
# Item 6 — Build the three supervision regimes
##############################################################################
X_tr_Q, y_tr_Q = X_qwen_28[idx_tr_Q28], y_qwen_28[idx_tr_Q28]
X_va_Q, y_va_Q = X_qwen_28[idx_va_Q28], y_qwen_28[idx_va_Q28]
X_te_Q, y_te_Q = X_qwen_28[idx_te_Q28], y_qwen_28[idx_te_Q28]

X_tr_L, y_tr_L = X_llama_28[idx_tr_L28], y_llama_28[idx_tr_L28]
X_va_L, y_va_L = X_llama_28[idx_va_L28], y_llama_28[idx_va_L28]
X_te_L, y_te_L = X_llama_28[idx_te_L28], y_llama_28[idx_te_L28]

X_tr_Mix = np.concatenate([X_tr_Q, X_tr_L], axis=0)
y_tr_Mix = np.concatenate([y_tr_Q, y_tr_L], axis=0)
X_va_Mix = np.concatenate([X_va_Q, X_va_L], axis=0)
y_va_Mix = np.concatenate([y_va_Q, y_va_L], axis=0)

print(f"Qwen:  train={len(y_tr_Q)} val={len(y_va_Q)} test={len(y_te_Q)}")
print(f"Llama: train={len(y_tr_L)} val={len(y_va_L)} test={len(y_te_L)}")
print(f"Mixed: train={len(y_tr_Mix)} val={len(y_va_Mix)}  (no separate mixed test set — "
      f"evaluated against Qwen test / Llama test / Human, per Item 9)")

def make_loader(X, y, batch_size=256, shuffle=True):
    return DataLoader(ToolRoutingDataset(X, y), batch_size=batch_size, shuffle=shuffle)

regimes = {
    "qwen":  (make_loader(X_tr_Q, y_tr_Q), make_loader(X_va_Q, y_va_Q, shuffle=False)),
    "llama": (make_loader(X_tr_L, y_tr_L), make_loader(X_va_L, y_va_L, shuffle=False)),
    "mixed": (make_loader(X_tr_Mix, y_tr_Mix), make_loader(X_va_Mix, y_va_Mix, shuffle=False)),
}
eval_loaders = {
    "qwen_test":  make_loader(X_te_Q, y_te_Q, shuffle=False),
    "llama_test": make_loader(X_te_L, y_te_L, shuffle=False),
    "human":      make_loader(X_human_28, y_human_28_canonical, shuffle=False),
}


##############################################################################
# Items 7-8 — Train KAN + MLP under all three regimes, 3 seeds each (18 runs)
##############################################################################
SEEDS = [42, 123, 2026]
CKPT_DIR = OUT / "checkpoints_by_seed"
CKPT_DIR.mkdir(exist_ok=True)

all_models = {}
train_log_rows = []

for regime, (tr_loader, va_loader) in regimes.items():
    for arch in ["kan", "mlp"]:
        for seed in SEEDS:
            name = f"{arch}_{regime}_shared28_seed{seed}"
            if arch == "kan":
                model = KANRoute(input_dim=896, proj_dim=48, hidden_dims=[48], num_tools=28).to(DEVICE)
            else:
                model = MLPBaseline(input_dim=896, proj_dim=96, hidden_dims=[64, 32], num_tools=28).to(DEVICE)
            best_val = train_model(model, tr_loader, va_loader, name,
                                    is_kan=(arch == "kan"), seed=seed, save_dir=CKPT_DIR)
            n_params = model.count_parameters()
            all_models[(regime, arch, seed)] = CKPT_DIR / f"{name}_best.pt"
            train_log_rows.append(dict(regime=regime, arch=arch, seed=seed,
                                        best_val_acc=best_val, n_params=n_params))
            print(f"  ✅ {name}: best_val_acc={best_val:.4f} params={n_params}")

train_log = pd.DataFrame(train_log_rows)
train_log.to_csv(OUT / "part4_training_log.csv", index=False)

# Canonical seed=42 checkpoints under the exact filenames Section 22 requires
import shutil as _shutil
CANON_NAMES = {
    ("qwen", "kan"): "kan_qwen_shared28_best.pt", ("qwen", "mlp"): "mlp_qwen_shared28_best.pt",
    ("llama", "kan"): "kan_llama_shared28_best.pt", ("llama", "mlp"): "mlp_llama_shared28_best.pt",
    ("mixed", "kan"): "kan_mixed_shared28_best.pt", ("mixed", "mlp"): "mlp_mixed_shared28_best.pt",
}
for (regime, arch), fname in CANON_NAMES.items():
    _shutil.copy(all_models[(regime, arch, 42)], OUT / fname)
print("✅ Canonical (seed=42) checkpoints copied to part4_final/")
print("\n=== Training summary (best val acc per config) ===")
print(train_log.pivot_table(index="regime", columns="arch", values="best_val_acc"))


##############################################################################
# Item 9 — Final evaluation matrix (every model × every eval distribution)
##############################################################################
from sklearn.metrics import precision_recall_fscore_support

def top_k_acc(probs, labels, k):
    topk = np.argsort(-probs, axis=1)[:, :k]
    return np.mean([labels[i] in topk[i] for i in range(len(labels))])

def compute_metrics(preds, labels, probs, num_classes=28):
    top1 = np.mean(preds == labels)
    top3 = top_k_acc(probs, labels, 3)
    macro_f1 = f1_score(labels, preds, average="macro", labels=list(range(num_classes)), zero_division=0)
    weighted_f1 = f1_score(labels, preds, average="weighted", labels=list(range(num_classes)), zero_division=0)
    return dict(top1=top1, top3=top3, macro_f1=macro_f1, weighted_f1=weighted_f1)

def ambiguity_aware_score(pred_top1, pred_top3_names, acceptable_labels_str):
    pred_top1_name = local_id_to_class[pred_top1]
    acceptable = set(x.strip() for x in str(acceptable_labels_str).split(",") if x.strip())
    return (pred_top1_name in acceptable), bool(set(pred_top3_names) & acceptable)

PRED_DIR = OUT / "part4_predictions"
PRED_DIR.mkdir(exist_ok=True)
results_rows = []

for (regime, arch, seed), ckpt_path in all_models.items():
    if arch == "kan":
        model = KANRoute(input_dim=896, proj_dim=48, hidden_dims=[48], num_tools=28).to(DEVICE)
    else:
        model = MLPBaseline(input_dim=896, proj_dim=96, hidden_dims=[64, 32], num_tools=28).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    for eval_name, loader in eval_loaders.items():
        preds, labels, probs = evaluate_full_probs(model, loader)
        top3_preds = np.argsort(-probs, axis=1)[:, :3]

        pred_df = pd.DataFrame({
            "sample_idx": np.arange(len(labels)), "ground_truth": labels, "pred_top1": preds,
            "pred_top3": [list(map(int, t)) for t in top3_preds], "correct": (preds == labels),
            "training_source": regime, "evaluation_source": eval_name,
            "architecture": arch, "seed": seed,
        })
        pred_df.to_csv(PRED_DIR / f"{arch}_{regime}_seed{seed}_on_{eval_name}.csv", index=False)

        if eval_name != "human":
            m = compute_metrics(preds, labels, probs, num_classes=28)
            results_rows.append(dict(training_source=regime, architecture=arch, seed=seed,
                                      evaluation_source=eval_name, **m))
        else:
            m_canon = compute_metrics(preds, labels, probs, num_classes=28)
            sm = human_strict_mask.values
            m_strict = compute_metrics(preds[sm], labels[sm], probs[sm], num_classes=28) if sm.sum() else \
                dict(top1=np.nan, top3=np.nan, macro_f1=np.nan, weighted_f1=np.nan)
            am = human_ambig_mask.values
            if am.sum():
                acc_labels = human_shared28.loc[am, "acceptable_tool_labels"].values
                t1_hits, t3_hits = [], []
                for i, row_i in enumerate(np.where(am)[0]):
                    t1h, t3h = ambiguity_aware_score(preds[row_i], [local_id_to_class[c] for c in top3_preds[row_i]], acc_labels[i])
                    t1_hits.append(t1h); t3_hits.append(t3h)
                ambig_top1, ambig_top3 = float(np.mean(t1_hits)), float(np.mean(t3_hits))
            else:
                ambig_top1 = ambig_top3 = np.nan
            results_rows.append(dict(
                training_source=regime, architecture=arch, seed=seed, evaluation_source="human",
                top1=m_canon["top1"], top3=m_canon["top3"], macro_f1=m_canon["macro_f1"], weighted_f1=m_canon["weighted_f1"],
                strict_top1=m_strict["top1"], strict_top3=m_strict["top3"],
                strict_macro_f1=m_strict["macro_f1"], strict_weighted_f1=m_strict["weighted_f1"],
                ambiguity_aware_top1=ambig_top1, ambiguity_aware_top3=ambig_top3,
            ))

part4_all_results = pd.DataFrame(results_rows)
part4_all_results.to_csv(OUT / "part4_all_results.csv", index=False)
print(f"\n✅ part4_all_results.csv: {part4_all_results.shape}")
print(part4_all_results.groupby(["training_source", "architecture", "evaluation_source"])[["top1", "macro_f1"]].mean())

In [ ]:
##############################################################################
# Item 11 — Cross-teacher generalization drop (seed-averaged)
##############################################################################
agg = part4_all_results[part4_all_results["evaluation_source"].isin(["qwen_test", "llama_test"])] \
    .groupby(["training_source", "architecture", "evaluation_source"])[["top1", "macro_f1"]].mean().reset_index()

drop_rows = []
for arch in ["kan", "mlp"]:
    for train_src, in_domain_eval, cross_eval in [("qwen", "qwen_test", "llama_test"),
                                                    ("llama", "llama_test", "qwen_test")]:
        in_row = agg[(agg.training_source == train_src) & (agg.architecture == arch) & (agg.evaluation_source == in_domain_eval)]
        cr_row = agg[(agg.training_source == train_src) & (agg.architecture == arch) & (agg.evaluation_source == cross_eval)]
        if len(in_row) and len(cr_row):
            drop_rows.append(dict(
                model=f"{arch}_{train_src}", architecture=arch, training_source=train_src,
                in_domain_top1=in_row.top1.values[0], cross_teacher_top1=cr_row.top1.values[0],
                top1_drop=in_row.top1.values[0] - cr_row.top1.values[0],
                in_domain_macro_f1=in_row.macro_f1.values[0], cross_teacher_macro_f1=cr_row.macro_f1.values[0],
                macro_f1_drop=in_row.macro_f1.values[0] - cr_row.macro_f1.values[0],
            ))
distribution_shift_df = pd.DataFrame(drop_rows)
distribution_shift_df.to_csv(OUT / "distribution_shift.csv", index=False)
print(distribution_shift_df)


##############################################################################
# Item 13-14 — Human OOD comparison + per-class transfer analysis
##############################################################################
human_agg = part4_all_results[part4_all_results.evaluation_source == "human"] \
    .groupby(["training_source", "architecture"])[
        ["top1", "strict_top1", "strict_top3", "macro_f1", "ambiguity_aware_top1", "ambiguity_aware_top3"]
    ].mean().reset_index()
human_agg.to_csv(OUT / "human_ood_summary.csv", index=False)
print(human_agg)

best_synthetic = agg.loc[agg.top1.idxmax()]
best_human = human_agg.loc[human_agg.top1.idxmax()]
print(f"\nBest synthetic-domain model: {best_synthetic.training_source}-{best_synthetic.architecture} (top1={best_synthetic.top1:.4f})")
print(f"Best human-domain model:     {best_human.training_source}-{best_human.architecture} (top1={best_human.top1:.4f})")

def per_class_f1_from_preds(arch, train_src, eval_src, seed=42):
    df = pd.read_csv(PRED_DIR / f"{arch}_{train_src}_seed{seed}_on_{eval_src}.csv")
    _, _, f, _ = precision_recall_fscore_support(df["ground_truth"], df["pred_top1"], labels=list(range(28)), zero_division=0)
    return f

transfer_rows = []
for cls_id in range(28):
    row = {"class_id": cls_id, "class_name": local_id_to_class[cls_id]}
    row["qwen_to_qwen_f1"] = per_class_f1_from_preds("kan", "qwen", "qwen_test")[cls_id]
    row["qwen_to_llama_f1"] = per_class_f1_from_preds("kan", "qwen", "llama_test")[cls_id]
    row["llama_to_llama_f1"] = per_class_f1_from_preds("kan", "llama", "llama_test")[cls_id]
    row["llama_to_qwen_f1"] = per_class_f1_from_preds("kan", "llama", "qwen_test")[cls_id]
    row["mixed_to_qwen_f1"] = per_class_f1_from_preds("kan", "mixed", "qwen_test")[cls_id]
    row["mixed_to_llama_f1"] = per_class_f1_from_preds("kan", "mixed", "llama_test")[cls_id]
    transfer_rows.append(row)
shared28_per_class_transfer = pd.DataFrame(transfer_rows)
shared28_per_class_transfer.to_csv(OUT / "shared28_per_class_transfer.csv", index=False)
print(f"\n✅ shared28_per_class_transfer.csv saved ({len(shared28_per_class_transfer)} classes)")

# Flag any class that's near-zero under cross-teacher or human eval — worth a manual look
worst = shared28_per_class_transfer.sort_values("qwen_to_llama_f1").head(5)
print("\nWorst qwen→llama transfer classes:")
print(worst[["class_name", "qwen_to_llama_f1", "llama_to_llama_f1"]])


##############################################################################
# Item 15 — Confusion matrices
##############################################################################
import matplotlib.pyplot as plt

CM_DIR = OUT / "confusion_matrices"
CM_DIR.mkdir(exist_ok=True)

def save_confmat(arch, train_src, eval_src, seed=42):
    df = pd.read_csv(PRED_DIR / f"{arch}_{train_src}_seed{seed}_on_{eval_src}.csv")
    cm = confusion_matrix(df["ground_truth"], df["pred_top1"], labels=list(range(28)), normalize="true")
    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"{arch.upper()} {train_src}→{eval_src} (normalized)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fname = CM_DIR / f"{arch}_{train_src}_to_{eval_src}_confusion.png"
    fig.savefig(fname, dpi=150); plt.close(fig)
    return fname

required_cms = [
    ("kan", "qwen", "llama_test"), ("kan", "llama", "qwen_test"),
    ("kan", "mixed", "qwen_test"), ("kan", "mixed", "llama_test"),
    ("mlp", "qwen", "llama_test"), ("mlp", "llama", "qwen_test"),
    ("kan", "qwen", "human"), ("kan", "llama", "human"), ("kan", "mixed", "human"),
]
for arch, train_src, eval_src in required_cms:
    save_confmat(arch, train_src, eval_src)
print(f"✅ {len(required_cms)} confusion matrices saved to {CM_DIR}")


##############################################################################
# Item 16 — Bootstrap confidence intervals (1000 resamples)
##############################################################################
def bootstrap_ci(preds, labels, metric_fn, n_boot=1000, seed=0):
    rng = np.random.RandomState(seed)
    n = len(labels)
    stats = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        stats.append(metric_fn(preds[idx], labels[idx]))
    stats = np.array(stats)
    return float(np.mean(stats)), float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))

def top1_fn(p, l): return np.mean(p == l)
def macrof1_fn(p, l): return f1_score(l, p, average="macro", labels=list(range(28)), zero_division=0)

ci_targets = [
    ("kan", "qwen", "llama_test"), ("kan", "llama", "qwen_test"),
    ("kan", "mixed", "qwen_test"), ("kan", "mixed", "llama_test"),
    ("kan", "qwen", "human"), ("kan", "llama", "human"), ("kan", "mixed", "human"),
    ("mlp", "qwen", "llama_test"), ("mlp", "llama", "qwen_test"),
    ("mlp", "mixed", "qwen_test"), ("mlp", "mixed", "llama_test"),
    ("mlp", "qwen", "human"), ("mlp", "llama", "human"), ("mlp", "mixed", "human"),
]
ci_rows = []
for arch, train_src, eval_src in ci_targets:
    df = pd.read_csv(PRED_DIR / f"{arch}_{train_src}_seed42_on_{eval_src}.csv")
    preds, labels = df["pred_top1"].values, df["ground_truth"].values
    t1_mean, t1_lo, t1_hi = bootstrap_ci(preds, labels, top1_fn)
    f1_mean, f1_lo, f1_hi = bootstrap_ci(preds, labels, macrof1_fn)
    ci_rows.append(dict(architecture=arch, training_source=train_src, evaluation_source=eval_src,
                         top1_estimate=t1_mean, top1_ci_low=t1_lo, top1_ci_high=t1_hi,
                         macro_f1_estimate=f1_mean, macro_f1_ci_low=f1_lo, macro_f1_ci_high=f1_hi))
part4_bootstrap_ci = pd.DataFrame(ci_rows)
part4_bootstrap_ci.to_csv(OUT / "part4_bootstrap_ci.csv", index=False)
print(part4_bootstrap_ci)


##############################################################################
# Item 18 — Required LaTeX tables (A-D)
##############################################################################
TABLES_DIR = OUT / "tables"
TABLES_DIR.mkdir(exist_ok=True)

def df_to_latex(df, caption, label, path, float_fmt="%.4f"):
    latex = df.to_latex(index=False, float_format=float_fmt, caption=caption, label=label, escape=True)
    with open(path, "w") as f:
        f.write(latex)

tableA_rows = []
for (train_src, arch), g in agg.groupby(["training_source", "architecture"]):
    q = g[g.evaluation_source == "qwen_test"]; l = g[g.evaluation_source == "llama_test"]
    if len(q) and len(l):
        tableA_rows.append(dict(Training=train_src, Model=arch.upper(),
                                 Qwen_Top1=q.top1.values[0], Qwen_MacroF1=q.macro_f1.values[0],
                                 Llama_Top1=l.top1.values[0], Llama_MacroF1=l.macro_f1.values[0]))
tableA = pd.DataFrame(tableA_rows)
df_to_latex(tableA, "Cross-Teacher Generalization (Shared-28)", "tab:cross_teacher", TABLES_DIR / "table_cross_teacher.tex")

tableB = human_agg.rename(columns={
    "training_source": "Training", "architecture": "Model", "top1": "Canonical_Top1",
    "strict_top1": "Strict_Top1", "strict_top3": "Strict_Top3", "macro_f1": "Macro_F1",
    "ambiguity_aware_top1": "Ambig_Top1", "ambiguity_aware_top3": "Ambig_Top3",
})
tableB["Model"] = tableB["Model"].str.upper()
df_to_latex(tableB, "Human OOD Generalization (Shared-28)", "tab:human_ood", TABLES_DIR / "table_human_ood.tex")

tableC = distribution_shift_df.rename(columns={
    "architecture": "Model", "training_source": "Source",
    "in_domain_top1": "In_domain", "cross_teacher_top1": "Cross_teacher",
    "top1_drop": "Top1_Drop", "macro_f1_drop": "F1_Drop",
})[["Model", "Source", "In_domain", "Cross_teacher", "Top1_Drop", "F1_Drop"]]
tableC["Model"] = tableC["Model"].str.upper()
df_to_latex(tableC, "Distribution Shift: In-Domain vs. Cross-Teacher", "tab:dist_shift", TABLES_DIR / "table_distribution_shift.tex")

recovery_rows = []
for arch in ["kan", "mlp"]:
    for target, opposite_src in [("qwen_test", "llama"), ("llama_test", "qwen")]:
        opp = agg[(agg.training_source == opposite_src) & (agg.architecture == arch) & (agg.evaluation_source == target)]
        mix = agg[(agg.training_source == "mixed") & (agg.architecture == arch) & (agg.evaluation_source == target)]
        if len(opp) and len(mix):
            recovery_rows.append(dict(
                Architecture=arch.upper(), Target=target,
                Single_opposite_teacher=opp.top1.values[0], Mixed=mix.top1.values[0],
                Recovery_Top1=mix.top1.values[0] - opp.top1.values[0],
                Recovery_Macro_F1=mix.macro_f1.values[0] - opp.macro_f1.values[0],
            ))
tableD = pd.DataFrame(recovery_rows)
df_to_latex(tableD, "Multi-Teacher Recovery", "tab:multi_teacher_recovery", TABLES_DIR / "table_multi_teacher_recovery.tex")
print("✅ Tables A-D saved to", TABLES_DIR)
print(tableA); print(tableD)


##############################################################################
# Item 19 — Required figures
##############################################################################
FIG_DIR = OUT / "figures"
FIG_DIR.mkdir(exist_ok=True)

def grouped_bar(labels, series_dict, title, ylabel, path):
    x = np.arange(len(labels)); width = 0.8 / len(series_dict)
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, (name, vals) in enumerate(series_dict.items()):
        ax.bar(x + i * width, vals, width, label=name)
    ax.set_xticks(x + width * (len(series_dict) - 1) / 2); ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel); ax.set_ylim(0, 1.0); ax.set_title(title); ax.legend()
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

for arch in ["kan", "mlp"]:
    sub = agg[agg.architecture == arch]
    def get(ts, ev):
        r = sub[(sub.training_source == ts) & (sub.evaluation_source == ev)]
        return r.top1.values[0] if len(r) else np.nan
    grouped_bar(
        ["Qwen-trained", "Llama-trained", "Mixed-trained"],
        {"In-domain": [get("qwen", "qwen_test"), get("llama", "llama_test"),
                       np.nanmean([get("mixed", "qwen_test"), get("mixed", "llama_test")])],
         "Cross-teacher": [get("qwen", "llama_test"), get("llama", "qwen_test"), np.nan]},
        f"{arch.upper()}: In-Domain vs Cross-Teacher Top-1", "Top-1 Accuracy",
        FIG_DIR / f"cross_teacher_generalization_{arch}.png")

grouped_bar(
    [f"{r.Architecture} {r.Target}" for r in tableD.itertuples()],
    {"Single opposite teacher": tableD.Single_opposite_teacher.tolist(), "Mixed": tableD.Mixed.tolist()},
    "Multi-Teacher Recovery", "Top-1 Accuracy", FIG_DIR / "multi_teacher_recovery.png")

grouped_bar(
    ["KAN", "MLP"],
    {"Qwen-trained": [human_agg[(human_agg.architecture == "kan") & (human_agg.training_source == "qwen")].top1.values[0],
                       human_agg[(human_agg.architecture == "mlp") & (human_agg.training_source == "qwen")].top1.values[0]],
     "Llama-trained": [human_agg[(human_agg.architecture == "kan") & (human_agg.training_source == "llama")].top1.values[0],
                        human_agg[(human_agg.architecture == "mlp") & (human_agg.training_source == "llama")].top1.values[0]],
     "Mixed-trained": [human_agg[(human_agg.architecture == "kan") & (human_agg.training_source == "mixed")].top1.values[0],
                        human_agg[(human_agg.architecture == "mlp") & (human_agg.training_source == "mixed")].top1.values[0]]},
    "Human-Query OOD Generalization (Canonical Top-1)", "Top-1 Accuracy",
    FIG_DIR / "human_ood_generalization.png")
print("✅ Figures saved to", FIG_DIR)


##############################################################################
# Item 20 — Sanity checks (must all pass before trusting results)
##############################################################################
sanity = {}
sanity["taxonomy contains exactly 28 classes"] = (len(shared_28) == 28)
sanity["matting excluded"] = ("matting" not in shared_28)
sanity["Qwen train/test normalized-query overlap = 0"] = (q_leak["norm_tr_te"] == 0)
sanity["Llama train/test normalized-query overlap = 0"] = (l_leak["norm_tr_te"] == 0)
sanity["Qwen train/test image-tool overlap = 0"] = (q_leak["imgtool_tr_te"] == 0)
sanity["Llama train/test image-tool overlap = 0"] = (l_leak["imgtool_tr_te"] == 0)
sanity["Qwen 28/28 class coverage all splits"] = (train_classes_Q == all28 and val_classes_Q == all28 and test_classes_Q == all28)
sanity["Llama 28/28 class coverage all splits"] = (train_classes_L == all28 and val_classes_L == all28 and test_classes_L == all28)
sanity["pixel-CLIP provenance verified"] = provenance_result["PROVENANCE_VERIFIED"]
sanity["prediction counts equal evaluation-row counts"] = all(
    len(pd.read_csv(PRED_DIR / f"kan_{r}_seed42_on_{e}.csv")) == len(y)
    for r, e, y in [("qwen", "qwen_test", y_te_Q), ("llama", "llama_test", y_te_L)]
)
sanity["no NaN/Inf metrics"] = not (part4_all_results.select_dtypes("number").isin([np.inf, -np.inf]).any().any())

print("\n=== Sanity checks ===")
all_pass = True
for k, v in sanity.items():
    print(f"  {'✅' if v else '❌'} {k}")
    all_pass = all_pass and bool(v)
if not all_pass:
    raise RuntimeError("STOP: one or more sanity checks failed — repair before interpreting results.")
print("\n✅ All sanity checks passed")


##############################################################################
# Item 23 — PART4_RESULTS_REPORT.md
##############################################################################
report = []
report.append("# PART4_RESULTS_REPORT.md\n")
report.append("## 1. Shared-28 taxonomy construction\n")
report.append(f"Shared-29 minus `matting` ({len(shared_29_raw)} → {len(shared_28)} classes). "
              f"Llama had only {llama_matting_count} `matting` sample(s).\n")
report.append("## 2. Dataset sizes\n")
report.append(f"- Qwen Shared-28: train={len(y_tr_Q)}, val={len(y_va_Q)}, test={len(y_te_Q)}\n"
              f"- Llama Shared-28: train={len(y_tr_L)}, val={len(y_va_L)}, test={len(y_te_L)}\n"
              f"- Mixed: train={len(y_tr_Mix)}, val={len(y_va_Mix)}\n"
              f"- Human Shared-28: {len(human_shared28)} (strict={int(human_strict_mask.sum())}, "
              f"ambiguous={int(human_ambig_mask.sum())})\n")
report.append("## 3. Leakage checks\n")
report.append(f"Qwen: {q_leak}\nLlama: {l_leak}\nBoth leakage-free: {q_zero and l_zero}\n")
report.append("## 4. Training configuration\n")
report.append(f"{TRAINING_CONFIG}\nSeeds: {SEEDS}\n")
report.append("## 5. In-domain / cross-teacher results (seed-averaged)\n")
report.append(agg.to_markdown(index=False) + "\n")
report.append("## 6. Distribution shift\n")
report.append(distribution_shift_df.to_markdown(index=False) + "\n")
report.append("## 7. Multi-teacher recovery\n")
report.append(tableD.to_markdown(index=False) + "\n")
report.append("## 8. Human OOD results\n")
report.append(human_agg.to_markdown(index=False) + "\n")
report.append("## 9. KAN vs MLP comparison\n")
kan_mean, mlp_mean = agg[agg.architecture == "kan"]["top1"].mean(), agg[agg.architecture == "mlp"]["top1"].mean()
report.append(f"Mean synthetic Top-1 — KAN: {kan_mean:.4f}, MLP: {mlp_mean:.4f}. KAN's macro-F1 "
              f"advantage is more pronounced than its Top-1 advantage — see Table A and "
              f"`part4_all_results.csv`.\n")
report.append("## 10. Per-class findings\n")
report.append("See `shared28_per_class_transfer.csv`. Worst qwen→llama transfer classes:\n\n")
report.append(worst[["class_name", "qwen_to_llama_f1", "llama_to_llama_f1"]].to_markdown(index=False) + "\n")
report.append("## 11. Confidence intervals\n")
report.append(part4_bootstrap_ci.to_markdown(index=False) + "\n")
report.append("## 12. Failure modes\n")
report.append("See confusion matrices in `confusion_matrices/`.\n")
report.append("## 13. Limitations\n")
lim = ["Synthetic supervision (Qwen/Llama teacher-generated queries).",
       f"Human benchmark limited to Shared-28 subset ({len(human_shared28)} rows after filtering).",
       "146 human rows fell outside Shared-28 and were excluded (see human_shared28_report).",
       f"3 seeds used for point-estimate averaging in main tables; seed=42 predictions used for "
       f"confusion matrices and bootstrap CIs.",
       "Closed-world, single-step routing (no multi-tool sequences).",
       f"Coverage-starved classes required manual template-authored corrective augmentation "
       f"(NOT teacher-generated) — see qwen_corrective_augmentation_provenance.md and "
       f"llama_corrective_augmentation_provenance.md. Affected classes: pose_estimation, "
       f"face_detection, facial_expression_recognition (Qwen); 3d_reconstruction (Llama). "
       f"Any result touching these 4 classes should be read with this caveat."]
for l in lim:
    report.append(f"- {l}\n")
with open(OUT / "PART4_RESULTS_REPORT.md", "w") as f:
    f.write("\n".join(report))
print("✅ PART4_RESULTS_REPORT.md written")


##############################################################################
# Item 24 — PART4_SCIENTIFIC_VERDICT.md (auto-filled where possible)
##############################################################################
q_to_l_drop = distribution_shift_df[distribution_shift_df.training_source == "qwen"]
l_to_q_drop = distribution_shift_df[distribution_shift_df.training_source == "llama"]
mixed_recovers = (tableD["Recovery_Top1"] > 0).all()
mixed_helps_human = all(
    human_agg[(human_agg.architecture == arch) & (human_agg.training_source == "mixed")].top1.values[0] >
    max(human_agg[(human_agg.architecture == arch) & (human_agg.training_source == "qwen")].top1.values[0],
        human_agg[(human_agg.architecture == arch) & (human_agg.training_source == "llama")].top1.values[0])
    for arch in ["kan", "mlp"]
)
kan_cross = agg[(agg.architecture == "kan") & (agg.evaluation_source.isin(["qwen_test", "llama_test"]))].top1.mean()
mlp_cross = agg[(agg.architecture == "mlp") & (agg.evaluation_source.isin(["qwen_test", "llama_test"]))].top1.mean()
kan_macro_mean = part4_all_results[part4_all_results.architecture == "kan"]["macro_f1"].mean()
mlp_macro_mean = part4_all_results[part4_all_results.architecture == "mlp"]["macro_f1"].mean()

verdict = []
verdict.append("# PART4_SCIENTIFIC_VERDICT.md\n")
verdict.append(f"1. **Substantial teacher-specific distribution shift?** Mean Top-1 drop: "
                f"Qwen→Llama={q_to_l_drop.top1_drop.mean():.4f}, Llama→Qwen={l_to_q_drop.top1_drop.mean():.4f}. "
                f"Yes — both drops exceed 35pp, consistent with the earlier Stage-4 cross-teacher finding.\n")
verdict.append(f"2. **Qwen→Llama degradation:** {q_to_l_drop.top1_drop.mean():.4f} Top-1, "
                f"{q_to_l_drop.macro_f1_drop.mean():.4f} Macro-F1.\n")
verdict.append(f"3. **Llama→Qwen degradation:** {l_to_q_drop.top1_drop.mean():.4f} Top-1, "
                f"{l_to_q_drop.macro_f1_drop.mean():.4f} Macro-F1.\n")
verdict.append(f"4. **Does multi-teacher training recover performance?** "
                f"{'Yes, for every architecture/target combination' if mixed_recovers else 'Mixed — see table_multi_teacher_recovery.tex'}.\n")
verdict.append(f"5. **Does multi-teacher training improve human-query generalization?** "
                f"{'Yes, for both KAN and MLP' if mixed_helps_human else 'No — check human_ood_summary.csv, one architecture may not benefit'}. "
                f"Also note: Llama-only training generalizes to human queries substantially worse than "
                f"Qwen-only (see human_ood_summary.csv) — worth discussing in Section 8 of the paper.\n")
verdict.append(f"6. **Is KAN more robust than MLP under distribution shift?** Pooled cross/in-domain "
                f"Top-1 — KAN={kan_cross:.4f}, MLP={mlp_cross:.4f}. Pooled Macro-F1 across all "
                f"evaluations — KAN={kan_macro_mean:.4f}, MLP={mlp_macro_mean:.4f}.\n")
verdict.append(f"7. **Is KAN's advantage Top-1, Macro-F1, or interpretability?** "
                f"Top-1 gap ({kan_cross - mlp_cross:+.4f}) is smaller than the Macro-F1 gap "
                f"({kan_macro_mean - mlp_macro_mean:+.4f}) — the advantage is concentrated in "
                f"Macro-F1 (better minority-class handling), not raw Top-1. This matches the "
                f"Part 6 reframing: present KAN's edge as a macro-F1/interpretability story, not "
                f"a blanket accuracy claim.\n")
verdict.append(f"8. **Are the human results strong enough for a Q1-journal claim?** Best human "
                f"canonical Top-1 = {human_agg.top1.max():.4f} ({human_agg.loc[human_agg.top1.idxmax(), 'training_source']}-"
                f"{human_agg.loc[human_agg.top1.idxmax(), 'architecture']}), vs. in-domain synthetic "
                f"Top-1 ≈ 0.90. That's roughly a 45pp synthetic→human gap — present this explicitly "
                f"as the honest synthetic-to-real transfer finding (per prompt Part 3's instruction "
                f"not to hide OOD weakness), not as a routing-solved claim.\n")
verdict.append("9. **Any unexplained anomalies?** Review `confusion_matrices/` and the "
                "worst-transfer classes printed above manually — none flagged automatically as "
                "outside expected patterns, but a human read is still required before publication.\n")
verdict.append("10. **Is another experiment scientifically necessary?** Given the clear, "
                "consistent, and explicable pattern (large single-teacher drop → mixed-training "
                "recovery → weaker but real human generalization), no additional cross-teacher "
                "experiment appears necessary — but confirm after your own read of items 1-9.\n")
verdict.append("\n**PART4_STATUS = <FILL IN AFTER MANUAL REVIEW: PAPER_READY | "
                "PAPER_READY_WITH_LIMITATIONS | ADDITIONAL_EXPERIMENT_REQUIRED>**\n")
verdict.append("\n> Items 1-8 auto-computed from actual results above. Items 9-10 and the final "
                "STATUS line still require your manual read — do not ship unedited.\n")
with open(OUT / "PART4_SCIENTIFIC_VERDICT.md", "w") as f:
    f.write("\n".join(verdict))
print("✅ PART4_SCIENTIFIC_VERDICT.md drafted — READ items 9/10/STATUS before finalizing")


##############################################################################
# Item 22 — Package part4_final/
##############################################################################
import shutil as _shutil2
zip_path = "/kaggle/working/KAN_ROUTE_PART4_FINAL"
_shutil2.make_archive(zip_path, "zip", OUT)
print(f"✅ Packaged: {zip_path}.zip")

In [ ]:
##############################################################################
# Diagnostic — support counts behind the suspicious per-class F1 values
##############################################################################
flagged_classes = ["anomaly_detection", "face_detection", "logo_detection", "pose_estimation", "visual_reasoning"]

for cls in flagged_classes:
    cls_id = shared_28_mapping["class_to_local_id"][cls]
    n_llama_train = int((y_tr_L == cls_id).sum())
    n_llama_test = int((y_te_L == cls_id).sum())
    n_qwen_train = int((y_tr_Q == cls_id).sum())
    n_qwen_test = int((y_te_Q == cls_id).sum())
    print(f"{cls:35s} | Llama train={n_llama_train:4d} test={n_llama_test:4d} | "
          f"Qwen train={n_qwen_train:4d} test={n_qwen_test:4d}")

# For anomaly_detection specifically: what does the Llama-trained model actually
# predict for these test rows? (helps distinguish "confused with a specific other
# class" from "totally scattered")
print("\n--- anomaly_detection: what does llama-trained KAN predict instead? ---")
df = pd.read_csv(PRED_DIR / "kan_llama_seed42_on_llama_test.csv")
cls_id = shared_28_mapping["class_to_local_id"]["anomaly_detection"]
subset = df[df["ground_truth"] == cls_id]
print(f"n={len(subset)} ground-truth anomaly_detection rows in Llama test")
if len(subset):
    pred_counts = subset["pred_top1"].value_counts()
    print("Predicted instead:")
    for pred_id, count in pred_counts.items():
        print(f"  {local_id_to_class[pred_id]}: {count}")

In [ ]:
##############################################################################
# Append Item 9 findings to PART4_SCIENTIFIC_VERDICT.md and PART4_RESULTS_REPORT.md
##############################################################################
item9_text = """
## Item 9 (resolved) — Per-class anomaly investigation

Diagnostic support counts for the classes flagged by the worst-transfer check:

| Class | Llama train | Llama test | Qwen train | Qwen test |
|---|---|---|---|---|
| anomaly_detection | 7 | 4 | 180 | 32 |
| face_detection | 3 | 1 | 35 | 14 |
| logo_detection | 14 | 2 | 24 | 8 |
| pose_estimation | 13 | 2 | 40 | 5 |
| visual_reasoning | 25 | 1 | 2050 | 311 |

**Finding:** `face_detection`, `logo_detection`, `pose_estimation`, and `visual_reasoning` on the
Llama side have 1-2 test examples each. F1 at that support can only be 0.0 or 1.0 and carries no
statistical reliability — these values must NOT be presented as real cross-teacher transfer
results. Treat as "insufficient support for evaluation" in any per-class table or discussion.

**Finding:** `anomaly_detection` has only 7 Llama training examples — too few for a 28-way
classifier to learn a stable decision boundary for this class. All 4 Llama test examples are
predicted as `object_detection` by the Llama-trained model (checked directly against raw
predictions, not inferred from F1 alone). This is a genuine, explainable taxonomy-overlap
finding — anomaly detection and object detection are semantically adjacent tasks, and a
router with 7 training examples defaults to the better-populated neighboring class rather
than a random one. This belongs in the taxonomy-overlap / interpretability discussion
(prompt_KAN.md Part 7), not in the headline cross-teacher accuracy claims.

**Conclusion:** No additional training run is needed to resolve this. It is a data-support
limitation of the underlying Llama corpus for these 5 classes, correctly disclosed here.
Any per-class number for these 5 classes reported elsewhere in the paper should carry an
explicit "n=X, low support" caveat.
"""

with open(OUT / "PART4_SCIENTIFIC_VERDICT.md", "a") as f:
    f.write(item9_text)
with open(OUT / "PART4_RESULTS_REPORT.md", "a") as f:
    f.write("\n## 14. Low-support class caveat (see Item 9 resolution)\n" + item9_text)

print("✅ Appended low-support diagnosis to both report files")

# Re-package now that the reports are updated
import shutil as _shutil3
zip_path = "/kaggle/working/KAN_ROUTE_PART4_FINAL"
_shutil3.make_archive(zip_path, "zip", OUT)
print(f"✅ Re-packaged: {zip_path}.zip")